<a href="https://colab.research.google.com/github/suryavedula/agentic_ai_kgp/blob/development/Copy_of_SuryaVedula.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Guessing to Grounded: Advanced Prompting and Sparse RAG

---

> ### Assignment
> You will turn an LLM from a **guesser** into a **grounded answerer** on a
> knowledge-intensive question-answering task - first through better
> **prompting**, then by adding **sparse retrieval (BM25 RAG)**.
>
> **Part 1: Advanced Prompting** takes a fixed task and improves it with
> prompting alone: **zero-shot → few-shot → chain-of-thought → self-consistency**.
> The model answers *closed-book* (from parametric memory only), so you will see
> exactly how far clever prompting can take a model that cannot look anything up.
>
> **Part 2: Sparse RAG** adds a **BM25 retriever**, grounds the model in
> retrieved passages, and then transfers the whole pipeline to a **domain corpus
> (HR)**.
>
> **Cells marked `# -- PROVIDED --` carry no marks** - just run them.
> **Cells marked `### CODE HERE ###` carry marks** - these are yours to complete.
> Reflection cells (`***your answer here***`) are part of your grade where a mark
> value is shown.
>
> Use `seed = 2026` wherever a seed is accepted. State the seed and `EVAL_N`
> you actually ran with in your final reflection.

---

## What You Will Build

| Section | Approach | Key Idea |
|---------|----------|----------|
| 3 | Zero-shot | Ask directly, no examples - the raw guesser |
| 4 | Few-shot | In-context examples fix the *format* and nudge behaviour |
| 5 | Chain-of-Thought | "Think step by step" - decompose multi-hop questions |
| 6 | Self-consistency | Sample several reasoning paths, take a majority vote |
| 8 | BM25 retrieval | Rank passages by weighted word overlap; free relevance labels |
| 9 | Grounded RAG | Inject top-k passages, force the model to answer from them |
| 10 | Ablations | How many passages? Oracle vs retrieved vs none |
| 11 | Domain transfer | Move the same pipeline to an HR corpus (HR-MultiWOZ) |

**Total Marks: 100**  (Part 1: 50 | Part 2: 50)

### Datasets
- **HotpotQA (distractor)** - multi-hop questions, each shipping **10 paragraphs
  (2 gold + 8 distractors)**. The gold paragraphs are labelled, giving us **free
  relevance labels** for evaluating retrieval.
- **HR-MultiWOZ (extractive QA)** - short HR conversation snippets with a
  question and an extractive answer; used for the domain-transfer stage.

### The Central Question
> *A multi-hop question like "The director of the 2014 film X was born in which
> city?" needs two facts chained together. A closed-book model often guesses a
> plausible-sounding wrong city. Can prompting alone fix this - and if not, does
> giving the model the right paragraphs?*

---


# PART 1: Advanced Prompting  *(50 marks)*

We work **closed-book**: the model sees only the **question** and must answer
from what it already knows. This isolates the effect of the *prompting technique*
from the effect of *retrieval* (which we add in Part 2).

## Setup  *(provided - run all cells, no marks)*

We use **Llama-3.2-3B-Instruct** through the HuggingFace `pipeline`, the same
model used throughout Module 2. A GPU runtime is strongly recommended
(Colab: *Runtime → Change runtime type → T4 GPU*).

In [ ]:
# -- PROVIDED --------------------------------------------------------------
# Run once. -q keeps the output quiet.
%pip install -q transformers accelerate datasets rank_bm25

In [ ]:
# -- PROVIDED --------------------------------------------------------------
import os, re, json, math, random, string, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
print("Imports ready.")

In [ ]:
# -- PROVIDED --------------------------------------------------------------
import torch
from transformers import pipeline, GenerationConfig

# Non-gated mirror of Llama-3.2-3B-Instruct (no HF token required).
# If you have gated access you may swap in "meta-llama/Llama-3.2-3B-Instruct".
MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"

pipe = pipeline(
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,   # use torch.float32 if your GPU lacks bf16
    device_map="auto",            # GPU if available, else CPU
)

tok = pipe.tokenizer
# Suppress the BPE tokenization spaces warning
tok.clean_up_tokenization_spaces = False

if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id

# To avoid the 'generation_config' conflict warning, we ensure the model config
# doesn't conflict with explicit parameters passed in the chat() function.
pipe.model.generation_config = GenerationConfig.from_model_config(pipe.model.config)
pipe.model.generation_config.pad_token_id = tok.eos_token_id

print("Loaded", MODEL_ID, "on", pipe.model.device)

In [ ]:
# -- PROVIDED --------------------------------------------------------------
def chat(user_prompt, system_prompt="You are a helpful assistant.",
         max_new_tokens=256, temperature=0.0):
    """Single-turn (system + user) call to Llama. Greedy if temperature == 0,
    otherwise nucleus sampling. Returns the assistant reply as a string."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    do_sample = temperature and temperature > 0

    # We remove max_length to avoid the conflict warning with max_new_tokens
    out = pipe(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=bool(do_sample),
        temperature=(temperature if do_sample else None),
        top_p=(0.9 if do_sample else None),
        generation_config=None # Passing None ensures individual params take precedence
    )
    return out[0]["generated_text"][-1]["content"].strip()

# A shared answer parser: every technique asks the model to end with "Answer: ..."
def extract_final_answer(text):
    """Return the text after the last 'Answer:' marker, else the last non-empty line."""
    matches = re.findall(r"(?i)answer\s*:\s*(.*)", text)
    if matches:
        ans = matches[-1].strip()
    else:
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        ans = lines[-1] if lines else text.strip()
    # strip surrounding quotes / trailing period
    return ans.strip().strip('"').strip("'").rstrip(".").strip()

# Smoke test (no task yet):
print(chat("In one sentence, what is retrieval-augmented generation?"))

## 1. Data Loading and Understanding  <font color="red">**[6 marks]**</font>

### 1.1 Load HotpotQA and build the working sets  <font color="red">[3 marks]</font>

**What to do:**
- Load `hotpotqa/hotpot_qa`, config `"distractor"` with `load_dataset`.
- Build an **evaluation set** of `EVAL_N = 50` examples from the `validation` split
  (use `.shuffle(seed=SEED).select(...)`), and a **few-shot pool** of 6 examples
  from the `train` split (kept separate so exemplars never leak into evaluation).
- Each HotpotQA example carries `question`, `answer`, `type`, `level`,
  `context = {"title": [...], "sentences": [[...], ...]}` (10 paragraphs), and
  `supporting_facts = {"title": [...], "sent_id": [...]}` (the **gold** titles).
- Write a helper `get_paragraphs(example)` that returns a list of
  `{"title": ..., "text": ...}` dicts (one per paragraph, sentences joined into
  one string), and `get_gold_titles(example)` returning the **set** of supporting
  titles. You will reuse both in Part 2.

In [ ]:
EVAL_N = 50
from datasets import load_dataset

hotpot = ### CODE HERE ###                      # load_dataset(..., "distractor")
eval_set     = ### CODE HERE ###                # 50 shuffled validation examples
fewshot_pool = ### CODE HERE ###                # 6 train examples (exemplars)

def get_paragraphs(example):
    """Return [{'title': str, 'text': str}, ...] - one entry per paragraph."""
    ### CODE HERE ###

def get_gold_titles(example):
    """Return the set of supporting-fact titles (the gold paragraphs)."""
    ### CODE HERE ###

print("eval:", len(eval_set), "| few-shot pool:", len(fewshot_pool))
print("paragraphs in example 0:", len(get_paragraphs(eval_set[0])))
print("gold titles in example 0:", get_gold_titles(eval_set[0]))

### 1.2 Understand the data  <font color="red">[3 marks]</font>

**What to do:** on the evaluation set, produce all of the following:
- A bar chart of **question type** (`bridge` vs `comparison`) and a bar chart of
  **level** (`easy`/`medium`/`hard`).
- Confirm numerically that **every** example has exactly **10 paragraphs** and
  **2 gold titles** (print the min/max of each).
- A histogram of **answer length in words**, with the mean and median printed.
- Print **one full example**: the question, the answer, the gold titles, and the
  first ~40 words of each gold paragraph (so you can see what "supporting" means).

In [ ]:
### CODE HERE ###
# 1) bar charts: question type, level
# 2) min/max number of paragraphs and gold titles
# 3) answer-length histogram + mean/median
# 4) print one full example (question, answer, gold titles, gold paragraph snippets)

### Interpretation Checkpoint  *(self-review, not scored)*

**[Your Answer]:** Answer each question in 2-4 sentences.

1. HotpotQA answers are usually very short (often 1-3 words). Why does this make **Exact Match** a meaningful metric here, and why do we still also want a token-level F1?
2. Each question ships 2 gold and 8 distractor paragraphs. Why are the distractors essential for evaluating a *retriever* (as opposed to just handing the model the gold paragraphs)?
3. These are multi-hop questions. In one sentence, why might a closed-book model get the *first* hop right but the final answer wrong?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***


## 2. The Evaluation Harness  <font color="red">**[6 marks]**</font>

Every technique in this assignment is scored with the **SQuAD-style** metrics:
**Exact Match (EM)** and **token-level F1**, both computed on *normalised* text
(lowercase, strip punctuation/articles/extra whitespace). Get these right once -
everything downstream depends on them.

### Why normalise?

The model might answer `"The Eiffel Tower."` when the gold answer is
`"Eiffel Tower"`. Without normalisation that is a mismatch. The standard SQuAD
normalisation lowercases, removes punctuation, drops the articles *a/an/the*, and
collapses whitespace before comparing.

- **Exact Match**: 1 if normalised prediction == normalised gold, else 0.
- **Token F1**: treat both as bags of tokens; F1 = harmonic mean of the precision
  and recall of shared tokens. Gives partial credit.

### 2.1 Implement the metrics  <font color="red">[6 marks]</font>

**What to do:** implement the three functions below.
- `normalize_text(s)` - lowercase; remove punctuation; remove articles `a`, `an`,
  `the`; collapse whitespace.
- `exact_match(pred, gold)` - 1.0 / 0.0 on normalised strings.
- `f1_token(pred, gold)` - token-level F1 over normalised tokens (return 0.0 if
  either side is empty; handle the yes/no case naturally).

In [ ]:
def normalize_text(s):
    """Lowercase, remove punctuation and articles, collapse whitespace."""
    ### CODE HERE ###

def exact_match(pred, gold):
    ### CODE HERE ###

def f1_token(pred, gold):
    ### CODE HERE ###

# Sanity checks (these should pass):
assert exact_match("The Eiffel Tower.", "eiffel tower") == 1.0
assert exact_match("Paris", "London") == 0.0
assert round(f1_token("University of Oxford", "Oxford"), 2) == 0.5
print("Metric checks passed.")

In [ ]:
# -- PROVIDED --------------------------------------------------------------
# A single harness used by EVERY technique. You write a predict_fn(example) -> str;
# this loops over the eval set, scores it, and stores the predictions for analysis.
RESULTS = {}   # name -> {"em":..., "f1":..., "preds":[...]}

def evaluate(predict_fn, examples, name, verbose_every=10):
    preds, ems, f1s = [], [], []
    for i, ex in enumerate(examples):
        pred = predict_fn(ex)
        gold = ex["answer"]
        preds.append(pred)
        ems.append(exact_match(pred, gold))
        f1s.append(f1_token(pred, gold))
        if verbose_every and i % verbose_every == 0:
            print(f"  [{i:>3}/{len(examples)}] pred={pred!r:<28} gold={gold!r}")
    em, f1 = 100 * np.mean(ems), 100 * np.mean(f1s)
    RESULTS[name] = {"em": em, "f1": f1, "preds": preds}
    print(f"==> {name}:  EM = {em:.1f}   F1 = {f1:.1f}")
    return RESULTS[name]

print("Harness ready. RESULTS dict initialised.")

## 3. Zero-Shot Prompting  <font color="red">**[7 marks]**</font>

### Zero-shot

Zero-shot means **no examples** - just an instruction and the question. The whole
burden is on the instruction. Two things matter a lot here:

1. **Constrain the output.** HotpotQA answers are short spans. If you don't say
   "answer in as few words as possible", the model writes a paragraph and EM
   collapses to ~0 even when the right span is buried inside.
2. **Pin the format.** Ask it to finish with `Answer: <answer>` so your parser
   (`extract_final_answer`, provided) can reliably pull the span out.

### 3.1 Build, run, and score the zero-shot prompt  <font color="red">[5 marks]</font>

**What to do:** write `predict_zeroshot(example)` that builds a zero-shot prompt
(a clear system prompt + a user prompt containing the question and a format
instruction), calls `chat(...)` **greedily** (`temperature=0.0`), and returns the
parsed answer via `extract_final_answer`. Then run it through `evaluate(...)`.

In [ ]:
ZS_SYSTEM = ### CODE HERE ###     # instruction: precise QA, short spans, end with "Answer:"

def predict_zeroshot(example):
    ### CODE HERE ###
    # build user prompt with the question + a "respond with Answer: <span>" instruction
    # call chat(...) greedily, parse with extract_final_answer, return the span

_ = evaluate(predict_zeroshot, eval_set, "Zero-shot")

### 3.2 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report your zero-shot EM and F1. Take 3 wrong predictions and check, for each, whether the model got the *first* hop right but missed the *second* - or whether it never had either fact. What does that split say about the closed-book ceiling?
2. Why is the gap between EM and F1 informative here? What would a large F1-EM gap tell you about what to fix first?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 4. Few-Shot Prompting  <font color="red">**[7 marks]**</font>

### Few-shot

Few-shot prompting prepends a handful of **worked examples** (drawn from the
*train* pool, never from the eval set) before the real question. The examples
teach the model the **task and the exact output format** by demonstration - this
usually fixes the verbose-answer problem from zero-shot and can nudge accuracy up.
Keep exemplars short and in the *same* `Question / Answer:` shape you want back.

### 4.1 Build, run, and score the few-shot prompt  <font color="red">[5 marks]</font>

**What to do:**
- Write `build_fewshot_block(pool, k)` that formats `k` exemplars from
  `fewshot_pool` as `Question: ...\nAnswer: ...` blocks.
- Write `predict_fewshot(example, k=3)` that prepends that block, then the real
  question, calls `chat(...)` greedily, and parses the answer.
- Run it through `evaluate(...)`.

In [ ]:
def build_fewshot_block(pool, k=3):
    """Format k exemplars as 'Question: ...\nAnswer: ...' separated by blank lines."""
    ### CODE HERE ###

def predict_fewshot(example, k=3):
    ### CODE HERE ###

print(build_fewshot_block(fewshot_pool, 2))   # peek at the exemplar block
_ = evaluate(predict_fewshot, eval_set, "Few-shot")

### 4.2 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report few-shot EM/F1 against your zero-shot numbers. Did few-shot help more on EM or on F1, and what does that pattern say about *what* the exemplars actually fixed?
2. You drew exemplars from the train split, not the eval split. Why does that separation matter, and what would be misleading about your results if you ignored it?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 5. Chain-of-Thought (CoT)  <font color="red">**[8 marks]**</font>

### Chain-of-Thought

HotpotQA questions are **multi-hop**: you must chain two facts. Asking for the
answer directly forces the model to do all the reasoning silently in one step -
where it often skips a hop. **Chain-of-thought** prompting asks the model to write
its reasoning *before* committing to an answer:

> *Let's think step by step.*

The reasoning tokens act as a scratchpad, so the model can resolve hop 1, then use
it for hop 2. The catch: the model now emits a paragraph, so your prompt **must**
still demand a final `Answer: <span>` line for the parser to find.

### 5.1 Build, run, and score zero-shot CoT  <font color="red">[6 marks]</font>

**What to do:** write `predict_cot(example)` that:
- builds a prompt instructing the model to reason step by step and then end with
  `Answer: <answer>` on its own line,
- calls `chat(...)` with enough `max_new_tokens` for the reasoning (e.g. 256),
  greedily,
- returns the parsed final answer.

Then run `evaluate(...)`. (Inspect one full raw generation to confirm the model
is actually reasoning and that your parser grabs the last `Answer:` line.)

In [ ]:
COT_SYSTEM = ### CODE HERE ###    # reasoning assistant; must finish with "Answer: <span>"

def predict_cot(example):
    ### CODE HERE ###

# Peek at one raw reasoning trace before scoring:
raw = chat(f"Question: {eval_set[0]['question']}\nLet's think step by step. End with 'Answer: <answer>'.",
           system_prompt="You are a careful reasoning assistant.", max_new_tokens=256)
print(raw, "\n---\nparsed:", extract_final_answer(raw))

_ = evaluate(predict_cot, eval_set, "Chain-of-Thought")

### 5.2 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report CoT EM/F1 vs zero-shot and few-shot. Did step-by-step reasoning help, hurt, or do nothing on this closed-book task - and why is that plausible given the questions are about *facts the model may not know*?
2. Find one example where the reasoning trace invents a fact it never had (a fabricated entity, date, or link between the two hops). Quote the invented step and explain why longer reasoning can make a closed-book model *more* confidently wrong.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 6. Self-Consistency  <font color="red">**[8 marks]**</font>

### Self-consistency

A single greedy CoT chain can take a wrong turn. **Self-consistency** samples
**several** reasoning chains at a non-zero temperature, extracts the final answer
from each, and returns the **majority vote**. The intuition: there are many wrong
answers but the correct answer tends to be reached by several independent chains,
so it wins the vote.

Cost is the trade-off: `N` samples means `N×` the generations. We use a small
`N` (e.g. 7) here.

### 6.1 Build, run, and score self-consistency  <font color="red">[6 marks]</font>

**What to do:** write `predict_self_consistency(example, n=7, temperature=0.8)` that:
- calls the CoT-style prompt `n` times with **sampling on** (`temperature>0`),
- parses each final answer with `extract_final_answer`,
- returns the **majority vote** over the *normalised* answers (break ties by the
  first-seen answer). *Tip:* vote on `normalize_text(ans)` but return a readable
  surface form.
- **also computes an agreement score** for that question - the share of the `n`
  samples that backed the winning answer (5 of 7 chains agreeing → `5/7 = 0.714`).
  Append it to the provided list `SC_AGREEMENT` so the scoring harness stays
  unchanged, then report the **mean agreement** over the eval set and the number
  of questions where all `n` chains agreed (agreement `= 1.0`).

Then run `evaluate(...)`. (This is the slowest cell - it makes `EVAL_N × n` calls.)

In [ ]:
SC_AGREEMENT = []   # -- PROVIDED -- one agreement score per evaluated question

def predict_self_consistency(example, n=7, temperature=0.8):
    ### CODE HERE ###
    # 1. generate n CoT samples with temperature>0
    # 2. parse each final answer
    # 3. majority-vote on normalized answers; return a readable form
    # 4. agreement = (votes for the winner) / n  ->  SC_AGREEMENT.append(agreement)

_ = evaluate(predict_self_consistency, eval_set, "Self-consistency", verbose_every=10)

### CODE HERE ###
# report mean agreement over SC_AGREEMENT and how many questions were unanimous (== 1.0)

### 6.2 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report self-consistency EM/F1 vs single-chain CoT, together with your **mean agreement score** and your unanimous-question count. Did voting help, and does the agreement level explain why it helped as much (or as little) as it did?
2. Self-consistency costs N× the compute. On this closed-book task, is that a good trade? When (in Part 2's grounded setting) would you expect voting to matter more or less?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 7. Part 1 Comparison  <font color="red">**[8 marks]**</font>

### 7.1 Comparison table and chart  <font color="red">[5 marks]</font>

**What to do:**
- Build a `pandas` DataFrame from `RESULTS` with one row per technique (Zero-shot,
  Few-shot, Chain-of-Thought, Self-consistency) and columns `EM`, `F1`. Sort by F1
  descending and display it.
- Draw a **grouped bar chart** comparing EM and F1 across the four techniques,
  with a legend and labelled axes.

In [ ]:
### CODE HERE ###
# 1) DataFrame from RESULTS (4 prompting techniques), columns EM & F1, sorted by F1
# 2) grouped bar chart (EM vs F1) across techniques with legend

### 7.2 Error analysis  <font color="red">[3 marks]</font>

**What to do:** using your **best** Part-1 technique, find **3 questions it still
gets wrong**. For each, print the question, the gold answer, the prediction, and
write **one sentence** naming the likely cause (missing fact / wrong hop / format).
End with a one-line takeaway: *what is the ceiling of prompting-only here, and why?*

In [ ]:
### CODE HERE ###
# pick best technique by F1; find 3 wrong examples; print Q / gold / pred + a cause each

### Part 1 Reflection  *(self-review, not scored)*

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Across the four techniques, what was the single biggest jump and what caused it (format vs reasoning vs knowledge)?
2. State, in one sentence with your own numbers, the ceiling that prompting-only hit on this task - the number Part 2 must beat.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


---
# PART 2: Sparse RAG with BM25  *(50 marks)*

Part 1 showed the ceiling of prompting a **closed-book** model. Now we let it
**look things up**. For each question, the 10 shipped paragraphs become a tiny
corpus; a **BM25** retriever ranks them, and we feed the top-k into the prompt so
the model answers from *evidence* instead of memory. Because HotpotQA labels the
gold paragraphs, we can also measure **how good the retrieval itself is**.

## Sparse retrieval and BM25

**Sparse** retrieval scores a document by **weighted word overlap** with the
query - no neural embeddings, no training. **BM25** is the workhorse. The score of
document $d$ for query $q$ is:

$$\text{score}(q,d)=\sum_{t \in q}\text{idf}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d)+k_1\left(1-b+b\,\dfrac{|d|}{\text{avgdl}}\right)}$$

where $f(t,d)$ is the count of term $t$ in $d$, $|d|$ is the document length,
$\text{avgdl}$ is the average length, and the Okapi IDF is:

$$\text{idf}(t)=\ln\!\left(\frac{N-\text{df}(t)+0.5}{\text{df}(t)+0.5}\right)$$

Two knobs: $k_1$ (term-frequency saturation) and $b$ (length normalisation).
We use the `rank_bm25` library (`BM25Okapi`, defaults $k_1=1.5$, $b=0.75$).

## 8. BM25 Retrieval over the Per-Question Context  <font color="red">**[14 marks]**</font>

Each question carries its own 10 paragraphs, so we build a **fresh BM25 index per
question** and rank those 10. The 2 gold titles give us the relevance labels.

### 8.1 Tokenizer + per-question retriever  <font color="red">[6 marks]</font>

**What to do:**
- Write `tokenize(text)` - lowercase and split on any non-alphanumeric character
  (same scheme used in the Module 2 RAG session).
- Write `retrieve_bm25(question, paragraphs, k)` that builds a `BM25Okapi` index
  over the paragraph **texts**, scores the tokenised question, and returns the
  top-k as a list of `{"title", "text", "score"}` (highest score first).

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    """Lowercase, split on any non-alphanumeric character."""
    ### CODE HERE ###

def retrieve_bm25(question, paragraphs, k=3):
    """Return top-k paragraphs as [{'title','text','score'}, ...], best first."""
    ### CODE HERE ###

# Quick check on example 0:
for r in retrieve_bm25(eval_set[0]["question"], get_paragraphs(eval_set[0]), k=3):
    print(f"{r['score']:.2f}  [{r['title']}]")
print("gold:", get_gold_titles(eval_set[0]))

### 8.2 Retrieval evaluation  <font color="red">[6 marks]</font>

**What to do:** with the gold titles as labels, compute over the whole eval set, for
`k ∈ {1, 2, 4, 6}`:
- **Recall@k** - fraction of gold titles that appear in the top-k.
- **Precision@k** - fraction of top-k that are gold.
- **nDCG@k** - normalised discounted cumulative gain with **binary** relevance
  (a paragraph is worth 1 if its title is gold, else 0):

$$\text{DCG@}k=\sum_{i=1}^{k}\frac{rel_i}{\log_2(i+1)}
\qquad
\text{IDCG@}k=\sum_{i=1}^{\min(k,\,|G|)}\frac{1}{\log_2(i+1)}
\qquad
\text{nDCG@}k=\frac{\text{DCG@}k}{\text{IDCG@}k}$$

  where $i$ is the 1-based rank in your ranked list and $|G|$ is the number of gold
  paragraphs for that question. Unlike recall, nDCG@k rewards putting gold paragraphs
  *near the top*; unlike MRR, it accounts for **both** gold paragraphs, not just the
  first one found. Average over the eval set.

Print a small table. (Helper `retrieval_metrics_at_k(examples, k)` returning the
three numbers is a clean way to organise this.)

In [ ]:
def dcg_at_k(rels):
    """Discounted cumulative gain of a ranked list of binary relevances."""
    ### CODE HERE ###

def retrieval_metrics_at_k(examples, k):
    """Return (recall@k, precision@k, ndcg@k) averaged over examples."""
    ### CODE HERE ###

# Sanity checks on the gain/discount arithmetic (these should pass):
assert round(dcg_at_k([1, 0, 0]), 3) == 1.0                    # gold first: no discount
assert round(dcg_at_k([0, 1, 0]), 3) == 0.631                  # gold second: 1/log2(3)
assert dcg_at_k([1, 1, 0]) > dcg_at_k([1, 0, 1])               # earlier gold ranks higher

rows = []
for k in [1, 2, 4, 6]:
    rec, prec, ndcg = retrieval_metrics_at_k(eval_set, k)
    rows.append({"k": k, "Recall@k": rec, "Precision@k": prec, "nDCG@k": ndcg})
display(pd.DataFrame(rows).round(3))

### 8.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. As k grows, Recall@k rises but Precision@k falls. Explain this trade-off in terms of what you feed the generator next: what is the cost of a high k for the *answering* step?
2. Compare your Recall@k and nDCG@k columns. Find a k where recall is high but nDCG@k is noticeably lower, and explain what that gap says about *where in the ranking* the gold paragraphs are landing - and why that matters once the passages are pasted into a prompt.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 9. Grounded RAG Answering  <font color="red">**[14 marks]**</font>

### Grounding the generator

RAG = **retrieve, then generate**. We inject the top-k passages into the prompt,
each tagged with an `[id]`, and instruct the model to answer **only** from them
(and to say "I don't know" otherwise, citing the `[id]`s it used). This is the
move that turns the guesser into a grounded answerer: the second-hop fact it
couldn't recall in Part 1 is now sitting in the context window.

### 9.1 RAG prompt + predict + score  <font color="red">[8 marks]</font>

**What to do:**
- Write `build_rag_prompt(question, passages)` that renders the passages as
  `[title] text` blocks followed by the question and an answer instruction.
- Write `predict_rag(example, k=4)` that retrieves top-k with `retrieve_bm25`,
  builds the prompt, calls `chat(...)` greedily with a **grounding system prompt**
  (answer only from context; end with `Answer: <span>`), and parses the answer.
- Run it through `evaluate(...)` under the name `"RAG (k=4)"`.

In [ ]:
RAG_SYSTEM = ### CODE HERE ###   # answer ONLY from numbered context; short span; end "Answer:"

def build_rag_prompt(question, passages):
    """Render passages as '[title] text' blocks, then the question + instruction."""
    ### CODE HERE ###

def predict_rag(example, k=4):
    ### CODE HERE ###

_ = evaluate(predict_rag, eval_set, "RAG (k=4)")

### 9.2 With vs without retrieval  <font color="red">[4 marks]</font>

**What to do:** write `with_vs_without(example)` that prints, for a single question,
the closed-book answer (`predict_zeroshot`) and the RAG answer (`predict_rag`) side
by side along with the gold answer and which titles were retrieved. Run it on **3**
eval questions where Part 1 was wrong, to show retrieval fixing them.

In [ ]:
def with_vs_without(example, k=4):
    ### CODE HERE ###
    # print question, gold, closed-book pred, retrieved titles, RAG pred

### CODE HERE ###  # call it on 3 questions that zero-shot got wrong

### 9.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report RAG (k=4) EM/F1 against your best Part-1 number. Of the Part-1 questions that were wrong, count how many RAG now gets right *and* how many it newly gets wrong. What does the second number tell you about injecting context?
2. RAG can still be wrong even when the gold passage was retrieved. Name two distinct failure modes of the *generation* step (not the retrieval step) you can see in your outputs.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 10. Ablations - How Much Does Retrieval Matter?  <font color="red">**[10 marks]**</font>

### 10.1 Effect of k  <font color="red">[5 marks]</font>

**What to do:** run RAG for `k ∈ {1, 2, 4, 6}` (reuse `predict_rag` with the right
`k`), record EM/F1 for each, store under names like `"RAG (k=1)"`, and plot EM and
F1 **vs k**. Comment in the chart title or a print on where it peaks.

In [ ]:
### CODE HERE ###
# for k in [1,2,4,6]: evaluate(lambda ex: predict_rag(ex, k=k), eval_set, f"RAG (k={k})")
# then line-plot EM and F1 vs k

### 10.2 Oracle vs retrieved vs none  <font color="red">[3 marks]</font>

**What to do:** isolate retrieval quality from generation quality by comparing
three context settings at fixed budget:
- **No context** - that's your Part-1 zero-shot (already in `RESULTS`).
- **BM25 top-4** - `predict_rag(ex, k=4)` (already / re-run).
- **Oracle (gold only)** - feed exactly the 2 gold paragraphs, no retrieval.

Implement `predict_oracle(example)` and evaluate it as `"Oracle (gold)"`, then
print the three EM/F1 numbers together.

In [ ]:
def predict_oracle(example):
    ### CODE HERE ###
    # build passages from ONLY the gold paragraphs, then answer with build_rag_prompt + chat

_ = evaluate(predict_oracle, eval_set, "Oracle (gold)", verbose_every=0)

for name in ["Zero-shot", "RAG (k=4)", "Oracle (gold)"]:
    if name in RESULTS:
        print(f"{name:>15}:  EM={RESULTS[name]['em']:.1f}  F1={RESULTS[name]['f1']:.1f}")

### 10.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. The gap between **Oracle** and **BM25 top-k** is the cost of imperfect *retrieval*; the gap between **Oracle** and a perfect score is the cost of imperfect *generation*. From your numbers, which is the bigger bottleneck in this pipeline?
2. Your k-sweep and your §8.2 Recall@k table peak at different k. Name the k you would ship, and explain why the best *retrieval* k need not be the best *answering* k.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 11. Domain Transfer to HR (HR-MultiWOZ)  <font color="red">**[10 marks]**</font>

HotpotQA gives each question its own 10 paragraphs. A real deployment has **one big
corpus** and must retrieve from *everything*. HR-MultiWOZ lets us test that: short
HR conversation snippets, each with a question and an extractive answer. We build a
**single global BM25 index** over all the snippets and run the same pipeline.

### 11.1 Load HR-MultiWOZ and build a global index  <font color="red">[4 marks]</font>

**What to do:**
- Load `xwjzds/extractive_qa_question_answering_hr`. Each row has `question`,
  `answer`, and `answer_context` (the snippet the answer comes from).
- Take a `HR_N = 50`-example evaluation slice (shuffled, seed `SEED`).
- Build the **corpus** = the list of **unique** `answer_context` strings across the
  slice, and a single `BM25Okapi` index over them (`hr_index`). Keep a parallel
  list `hr_corpus` so you can map a retrieved row index back to its text.
- For retrieval evaluation, note each question's **gold context** is its own
  `answer_context`.

In [ ]:
HR_N = 50
hr_ds = ### CODE HERE ###                     # load_dataset(...)
hr_split = ### CODE HERE ###                  # train split? inspect .keys() first
hr_eval  = ### CODE HERE ###                  # HR_N shuffled examples

hr_corpus = ### CODE HERE ###                 # unique answer_context strings
hr_index  = ### CODE HERE ###                 # BM25Okapi over tokenized hr_corpus

print("HR eval:", len(hr_eval), "| corpus size:", len(hr_corpus))
print("example Q:", hr_eval[0]["question"])
print("gold answer:", hr_eval[0]["answer"])

### 11.2 Run RAG on HR + measure  <font color="red">[4 marks]</font>

**What to do:**
- Write `retrieve_hr(question, k)` that ranks `hr_corpus` with `hr_index` and
  returns the top-k as `{"title","text","score"}` (use the snippet index as a
  pseudo-title, e.g. `"ctx_<i>"`).
- Write `predict_rag_hr(example, k=4)` reusing `build_rag_prompt` + `RAG_SYSTEM`.
- Evaluate answer EM/F1 over `hr_eval` (store as `"RAG-HR (k=4)"`), and also report
  **Recall@4**: fraction of questions whose own `answer_context` is in the top-4.

In [ ]:
def retrieve_hr(question, k=4):
    ### CODE HERE ###

def predict_rag_hr(example, k=4):
    ### CODE HERE ###

# Answer quality:
preds_hr, ems, f1s = [], [], []
for ex in hr_eval:
    p = predict_rag_hr(ex)
    preds_hr.append(p)
    ems.append(exact_match(p, ex["answer"])); f1s.append(f1_token(p, ex["answer"]))
RESULTS["RAG-HR (k=4)"] = {"em": 100*np.mean(ems), "f1": 100*np.mean(f1s), "preds": preds_hr}
print(f"RAG-HR (k=4): EM={RESULTS['RAG-HR (k=4)']['em']:.1f} F1={RESULTS['RAG-HR (k=4)']['f1']:.1f}")

# Retrieval Recall@4 (gold context = the row's own answer_context):
### CODE HERE ###

### 11.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. The pipeline barely changed between HotpotQA and HR - only the corpus and loader. What does that say about RAG as an *architecture* for moving an LLM into a new domain, compared with fine-tuning the model on HR data?
2. Retrieving from one global HR corpus is harder than ranking 10 per-question paragraphs. Give one reason HR retrieval might be *easier* in practice and one reason it might be *harder*, referencing your HR Recall@4.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***


## 12. Final Comparison  <font color="red">**[2 marks]**</font>

### 12.1 Master table and chart  <font color="red">[2 marks]</font>

**What to do:** build one table comparing the headline numbers - the **best Part-1
prompting** technique, **RAG (best k)** on HotpotQA, **Oracle (gold)**, and
**RAG-HR (k=4)** - with EM and F1. Plot them as a grouped bar chart. This is the
"guesser → grounded answerer" story in one picture.

In [ ]:
### CODE HERE ###
# assemble a summary DataFrame across the headline settings + grouped bar chart

## Final Reflection  *(self-review, not scored)*

**[Your Answer]:** Answer each question in 2-4 sentences.

1. State, with your own numbers, the full arc: closed-book prompting ceiling → BM25 RAG on HotpotQA → Oracle. What was the single largest jump and why?
2. BM25 is lexical. Name one concrete query where a *dense* (embedding) retriever would beat BM25, and one where BM25 is the better choice.
3. Your RAG pipeline answers 'I don't know' when the context lacks the answer. Why is that behaviour valuable in the HR deployment specifically, beyond just the EM/F1 score?
4. If you had to improve this system next, would you spend effort on the retriever or the generator? Justify with the Oracle-vs-RAG gap you measured.
5. Give one risk of RAG that prompting alone did not have (think about what is now in the context window and where it comes from).

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

4. ***your answer here***

5. ***your answer here***


---
## Mark Summary

| Section | Topic | What you write | Marks |
|---------|-------|----------------|-------|
| **Part 1** | **Advanced Prompting** | | **50** |
| Setup | Model + chat helper | *Provided - run only* | 0 |
| 1 | Data loading + understanding | Code | 6 |
| 2 | Evaluation harness (EM / token-F1) | Code | 6 |
| 3 | Zero-shot (build + run + reflect) | Code + answers | 7 |
| 4 | Few-shot (build + run + reflect) | Code + answers | 7 |
| 5 | Chain-of-Thought (build + run + reflect) | Code + answers | 8 |
| 6 | Self-consistency (build + run + reflect) | Code + answers | 8 |
| 7 | Comparison + error analysis | Code | 8 |
| **Part 2** | **Sparse RAG (BM25)** | | **50** |
| 8 | BM25 retriever + retrieval metrics | Code + answers | 14 |
| 9 | Grounded RAG answering | Code + answers | 14 |
| 10 | Ablations (k sweep, oracle vs none) | Code + answers | 10 |
| 11 | Domain transfer to HR-MultiWOZ | Code + answers | 10 |
| 12 | Final comparison | Code | 2 |
| | **Total** | | **100** |

---
*End of Assignment*
